## Silver — municipios (DTB)
**Origem:** `workspace.bronze.dtb` → **Destino:** `workspace.silver.municipios`

- **Grão:** 1 linha por município (deduplicado por `codigo_municipio` = `codigo_municipio_completo` 7 díg)
- **Bronze cols (linha 7, 9 cols):** `uf` (**código** 2 díg, não sigla) | `nome_uf` | `regiao_geografica_intermediaria` (cód 4 díg) | `nome_regiao_geografica_intermediaria` | `regiao_geografica_imediata` (cód 6 díg) | `nome_regiao_geografica_imediata` | `municipio` (cód base) | `codigo_municipio_completo` (PK 7 díg) | `nome_municipio`
- **Transformações:** `codigo_municipio` = lpad(`codigo_municipio_completo`,7), `codigo_uf` = lpad(`uf`,2), `sigla_uf` = **mapa código UF → sigla** (uf contém código), `codigo/nome_regiao_*` derivados de `regiao_geografica_*` com tratamento de `nan`/`double` → string pad (4/6 díg), Title Case, deduplica PK.
- **Linhagem:** `bronze.dtb` (ODS header linha 7) → limpeza → `silver.municipios`
- **Qualidade:** nulos/duplicatas PK, nulos em região, mismatch `municipio` vs completo, count por `sigla_uf`.

In [0]:
%run ./_setup_dtb

In [0]:
from pyspark.sql import functions as F
from data_pipeline import save_table, add_column_comments
from metadata.metadata import SILVER_MUNICIPIOS_COMMENTS

In [0]:
SOURCE_TABLE = "workspace.bronze.dtb"
TARGET_TABLE = "workspace.silver.municipios"

# Mapeamento código UF -> sigla/nome/região (IBGE) — usado se bronze não trouxer essas colunas
UF_MAP = {
    "11": ("RO", "Rondônia", "Norte"),
    "12": ("AC", "Acre", "Norte"),
    "13": ("AM", "Amazonas", "Norte"),
    "14": ("RR", "Roraima", "Norte"),
    "15": ("PA", "Pará", "Norte"),
    "16": ("AP", "Amapá", "Norte"),
    "17": ("TO", "Tocantins", "Norte"),
    "21": ("MA", "Maranhão", "Nordeste"),
    "22": ("PI", "Piauí", "Nordeste"),
    "23": ("CE", "Ceará", "Nordeste"),
    "24": ("RN", "Rio Grande do Norte", "Nordeste"),
    "25": ("PB", "Paraíba", "Nordeste"),
    "26": ("PE", "Pernambuco", "Nordeste"),
    "27": ("AL", "Alagoas", "Nordeste"),
    "28": ("SE", "Sergipe", "Nordeste"),
    "29": ("BA", "Bahia", "Nordeste"),
    "31": ("MG", "Minas Gerais", "Sudeste"),
    "32": ("ES", "Espírito Santo", "Sudeste"),
    "33": ("RJ", "Rio de Janeiro", "Sudeste"),
    "35": ("SP", "São Paulo", "Sudeste"),
    "41": ("PR", "Paraná", "Sul"),
    "42": ("SC", "Santa Catarina", "Sul"),
    "43": ("RS", "Rio Grande do Sul", "Sul"),
    "50": ("MS", "Mato Grosso do Sul", "Centro-Oeste"),
    "51": ("MT", "Mato Grosso", "Centro-Oeste"),
    "52": ("GO", "Goiás", "Centro-Oeste"),
    "53": ("DF", "Distrito Federal", "Centro-Oeste"),
}
REGIAO_COD = {"Norte": "1", "Nordeste": "2", "Sudeste": "3", "Sul": "4", "Centro-Oeste": "5"}

In [0]:
df_bronze = spark.table(SOURCE_TABLE)
print(f"Bronze: {df_bronze.count():,} linhas, colunas={df_bronze.columns}")
display(df_bronze.limit(5))

In [0]:
# Bronze header linha 7 (normalizado): uf (CÓDIGO 2 díg) | nome_uf | regiao_geografica_intermediaria (4 díg) | nome_regiao_geografica_intermediaria | regiao_geografica_imediata (6 díg) | nome_regiao_geografica_imediata | municipio | codigo_municipio_completo | nome_municipio
# Fixes: uf é código -> sigla via mapa; região estava nula por cast double/nan -> trata com regexp e lpad 4/6.
def clean_str(col):
    return F.trim(F.regexp_replace(F.initcap(F.trim(col.cast("string"))), r"\\s+", " "))

def clean_codigo(col, width):
    # converte double/"nan"/espaços -> string limpa com zero-pad; evita "3101.0" e "nan"
    s = F.trim(F.col(col).cast("string"))
    s = F.regexp_replace(s, r"\.0$", "")
    s = F.when(s.isin("nan", "None", "null", ""), None).otherwise(s)
    return F.lpad(s, width, "0")

df = df_bronze

# 1) PK: codigo_municipio 7 dígitos a partir de codigo_municipio_completo
df = df.withColumn("codigo_municipio", F.lpad(F.regexp_replace(F.trim(F.col("codigo_municipio_completo").cast("string")), r"\.0$", ""), 7, "0"))
df = df.filter(F.col("codigo_municipio").rlike("^[0-9]{7}$"))

# 2) codigo_uf: lpad(uf,2) ou substr do codigo_municipio se uf nulo
df = df.withColumn("codigo_uf", clean_codigo("uf", 2))
df = df.withColumn("codigo_uf", F.when(F.col("codigo_uf").isNull(), F.substring(F.col("codigo_municipio"), 1, 2)).otherwise(F.col("codigo_uf")))

# 3) sigla_uf via mapa código->sigla (NÃO upper(uf))
sigla_expr = None
for cod, (sigla, nome_uf_val, regiao_val) in UF_MAP.items():
    cond = F.col("codigo_uf") == cod
    sigla_expr = F.when(cond, F.lit(sigla)) if sigla_expr is None else sigla_expr.when(cond, F.lit(sigla))
df = df.withColumn("sigla_uf", sigla_expr)

df = df.withColumn("nome_uf", clean_str(F.col("nome_uf")))
df = df.withColumn("nome_municipio", clean_str(F.col("nome_municipio")))

# 4) Regiões — estavam nulas por inferência double/nan; corrige com clean_codigo + clean_str
df = df.withColumn("codigo_regiao_geografica_intermediaria", clean_codigo("regiao_geografica_intermediaria", 4))
df = df.withColumn("nome_regiao_geografica_intermediaria", clean_str(F.col("nome_regiao_geografica_intermediaria")))
df = df.withColumn("codigo_regiao_geografica_imediata", clean_codigo("regiao_geografica_imediata", 6))
df = df.withColumn("nome_regiao_geografica_imediata", clean_str(F.col("nome_regiao_geografica_imediata")))

# 5) Validação auxiliar: campo 'municipio' vs prefixo do completo
df = df.withColumn("municipio_str", F.lpad(F.regexp_replace(F.trim(F.col("municipio").cast("string")), r"\.0$", ""), 6, "0"))
mismatch = df.filter(F.substring(F.col("codigo_municipio"), 1, 6) != F.col("municipio_str")).count()
if mismatch > 0:
    print(f"Aviso: {mismatch} linhas onde 'municipio' diverge do prefixo de codigo_municipio_completo")
nulos_regiao = df.filter(F.col("codigo_regiao_geografica_intermediaria").isNull() | F.col("codigo_regiao_geografica_imediata").isNull()).count()
if nulos_regiao > 0:
    print(f"Aviso: {nulos_regiao} linhas com região nula após limpeza (ver bronze)")

# 6) Seleção final — contrato silver.municipios (com região corrigida + sigla_uf) + deduplica
keep = ["codigo_municipio", "codigo_uf", "sigla_uf", "nome_uf", "codigo_regiao_geografica_intermediaria", "nome_regiao_geografica_intermediaria", "codigo_regiao_geografica_imediata", "nome_regiao_geografica_imediata", "nome_municipio"]
df = df.select(*keep).dropDuplicates(["codigo_municipio"])

print(f"Silver preview: {df.count():,} municípios")
display(df.limit(10))

In [0]:
save_table(df, TARGET_TABLE)
existing = set(spark.table(TARGET_TABLE).columns)
filtered = {k: v for k, v in SILVER_MUNICIPIOS_COMMENTS.items() if k in existing}
if filtered:
    add_column_comments(spark, TARGET_TABLE, filtered)

In [0]:
total = spark.table(TARGET_TABLE).count()
distintos = spark.table(TARGET_TABLE).select("codigo_municipio").distinct().count()
nulos = spark.table(TARGET_TABLE).filter(F.col("codigo_municipio").isNull()).count()
print(f"Total: {total:,} | Distintos codigo_municipio: {distintos:,} | Nulos PK: {nulos:,}")
print(f"Duplicatas PK: {total - distintos:,}")
# Distribuição por UF para validar total IBGE ~5.569
display(spark.sql(f"SELECT sigla_uf, nome_uf, count(*) as qtd FROM {TARGET_TABLE} GROUP BY sigla_uf, nome_uf ORDER BY sigla_uf"))
display(spark.sql(f"DESCRIBE TABLE {TARGET_TABLE}"))